## LIB

In [ ]:
# === LIB ===
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.optimize import minimize
from datetime import datetime as dt

from nelson_siegel_svensson import NelsonSiegelSvenssonCurve
from nelson_siegel_svensson.calibrate import calibrate_nss_ols

# ---------------- Heston risk–neutral senza lambda ----------------
def heston_charfunc(phi, S0, v0, kappaQ, thetaQ, xi, rho, tau, r):
    a = kappaQ * thetaQ
    b = kappaQ                      # niente lambda in Q
    rspi = rho * xi * phi * 1j
    d = np.sqrt((rspi - b)**2 + (phi*1j + phi**2) * xi**2)
    g = (b - rspi + d) / (b - rspi - d)

    exp1 = np.exp(r * phi * 1j * tau)
    term2 = S0**(phi*1j) * ((1 - g*np.exp(d*tau)) / (1 - g))**(-2*a/xi**2)
    exp2 = np.exp(a*tau*(b - rspi + d)/xi**2 + v0*(b - rspi + d)*((1 - np.exp(d*tau))/(1 - g*np.exp(d*tau)))/xi**2)
    return exp1 * term2 * exp2

def _integrand(phi, S0, K, v0, kappaQ, thetaQ, xi, rho, tau, r):
    args = (S0, v0, kappaQ, thetaQ, xi, rho, tau, r)
    num = np.exp(r*tau)*heston_charfunc(phi-1j, *args) - K*heston_charfunc(phi, *args)
    den = 1j*phi*K**(1j*phi)
    return num/den  # complesso

def heston_price_rec(S0, K, v0, kappaQ, thetaQ, xi, rho, tau, r, umax=100.0, N=10000):
    args = (S0, v0, kappaQ, thetaQ, xi, rho, tau, r)
    P = 0.0
    dphi = umax / N
    for i in range(1, N):
        phi = dphi * (2*i + 1) / 2
        num = np.exp(r*tau)*heston_charfunc(phi-1j, *args) - K*heston_charfunc(phi, *args)
        den = 1j*phi*K**(1j*phi)
        P += dphi * num/den
    return float(np.real((S0 - K*np.exp(-r*tau))/2 + P/np.pi))

def heston_price(S0, K, v0, kappaQ, thetaQ, xi, rho, tau, r):
    real_val, err = quad(lambda u: np.real(_integrand(u, S0, K, v0, kappaQ, thetaQ, xi, rho, tau, r)),
                         0.0, 100.0, limit=200)
    return float((S0 - K*np.exp(-r*tau))/2 + real_val/np.pi)

# --- test rapido ---
if __name__ == "__main__":
    S0, K = 100.0, 100.0
    v0, r = 0.1, 0.03
    kappaQ, thetaQ, xi, rho = 1.5768, 0.0398, 0.3, -0.5711
    tau = 1.0
    price = heston_price(S0, K, v0, kappaQ, thetaQ, xi, rho, tau, r)
    print("[TEST HESTON Q, no lambda] price =", price)


## RATE

https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve&field_tdr_date_value=202604

In [ ]:
# === RATE MANUALE === 
yield_maturities = np.array([1/12, 1.5/12, 2/12, 3/12, 4/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30])
yields = np.array([3.72, 3.71, 3.71, 3.68, 3.76, 3.71, 3.72, 3.88, 3.91, 4.02, 4.20, 4.40, 4.97, 4.98]).astype(float) / 100
curve_fit, status = calibrate_nss_ols(yield_maturities, yields)

print(curve_fit)


## IB

In [ ]:
# === IB OPTION CHAIN AMPIA SU GLD, ANCHE IN DIFFERITA ===
import os
import json, math, random, re, time
from typing import Iterable, List, Tuple, Optional
import numpy as np
import pandas as pd
from ib_insync import IB, Stock, Option, util, BarData

# =========================
# CONFIG
# =========================
TICKER = "GLD"
EXCHANGE = "SMART"
CURRENCY = "USD"
NY_TZ = "America/New_York"

OUT_PREFIX = "gld_chain_wide"
DATA_DIR = "Data"

# True = forza delayed market data
USE_DELAYED_ONLY = True

# Scadenze:
# "all"              -> tutte le scadenze che IB restituisce
# "dense_short_long" -> tutte le scadenze corte + campionamento sul lungo
EXPIRY_MODE = "dense_short_long"

# Se EXPIRY_MODE = dense_short_long:
SHORT_EXPIRY_DAYS_FULL = 75      # prendi tutte le scadenze entro 75 giorni
LONG_EXPIRY_SAMPLED = 18         # oltre i 75 giorni, campiona 18 scadenze distribuite fino al lungo

# Finestra strike:
# il codice seleziona strike simmetrici sotto/sopra il prezzo spot
NEAR_STRIKES_EACH_SIDE = 45      # strike vicini all'ATM per lato
FAR_STRIKES_EACH_SIDE = 25       # strike lontani per lato
INCLUDE_EXTREME_STRIKES = True   # includi anche gli strike più estremi della chain

# batch market data
BATCH_SIZE = 24
PASSES_PER_BATCH = 3
SLEEP_SECONDS = 6.0
PAUSE_BETWEEN_BATCHES = 0.25

# generic ticks:
# 106 = option computation / greeks
# 100 = option volume/open interest (spesso non sempre disponibile)
# 101 = option open interest
# 104 = historical volatility
# 106 è il più utile; gli altri possono rimanere o essere tolti
GENERIC_TICKS = "100,101,104,106"

# Se vuoi date manuali:
USE_ALL_EXPIRIES = True
DATES_IT = []

# =========================
# UTILS
# =========================
_IT_MONTH = {
    "gen": 1, "feb": 2, "mar": 3, "apr": 4, "mag": 5, "giu": 6,
    "lug": 7, "ago": 8, "set": 9, "ott": 10, "nov": 11, "dic": 12
}

def _to_ny(ts):
    t = pd.Timestamp(ts)
    return t.tz_localize(NY_TZ) if t.tzinfo is None else t.tz_convert(NY_TZ)

def yearfrac_365(start_ts, end_ts) -> float:
    s, e = _to_ny(start_ts), _to_ny(end_ts)
    return (e - s).total_seconds() / (365.0 * 24 * 3600.0)

def _finite(x) -> bool:
    try:
        return math.isfinite(float(x))
    except Exception:
        return False

def _finite_pos(x) -> bool:
    try:
        return math.isfinite(float(x)) and float(x) > 0
    except Exception:
        return False

def _f(x):
    try:
        v = float(x)
        return v if math.isfinite(v) else np.nan
    except Exception:
        return np.nan

def parse_user_expiries_it(user_expiries_it: Iterable[str]) -> List[pd.Timestamp]:
    out = []
    for s in user_expiries_it:
        s = s.strip().lower().replace("’", "'").replace("  ", " ")
        m = re.match(r"(\d{1,2})\s+([a-z]{3})\s+(\d{2,4})", s)
        if not m:
            continue
        d, mon, y = int(m.group(1)), _IT_MONTH[m.group(2)], int(m.group(3))
        if y < 100:
            y += 2000
        ts = _to_ny(pd.Timestamp(year=y, month=mon, day=d)) + pd.Timedelta(hours=16)
        out.append(ts)
    return sorted(set(out))

def _parse_expiry_to_ny_close(raw: str) -> Optional[pd.Timestamp]:
    s = re.sub(r"[^0-9]", "", str(raw or "").strip())
    if not s:
        return None
    if len(s) == 8:
        dt = pd.to_datetime(s, format="%Y%m%d", errors="coerce")
    elif len(s) == 6:
        dt = pd.to_datetime(s, format="%Y%m", errors="coerce")
    else:
        dt = pd.to_datetime(s, errors="coerce")
    if pd.isna(dt):
        return None
    return _to_ny(dt) + pd.Timedelta(hours=16)

def _sample_evenly(seq: List, n: int) -> List:
    if n <= 0 or not seq:
        return []
    if len(seq) <= n:
        return list(seq)
    idx = np.linspace(0, len(seq) - 1, n)
    idx = np.unique(np.round(idx).astype(int))
    return [seq[i] for i in idx]

# =========================
# IB SETUP
# =========================
SUPPRESS_IB_CODES = {10089, 10090, 10091, 10167, 2104}

def setup_marketdata_mode(ib: IB, delayed_only: bool = USE_DELAYED_ONLY):
    def _quiet(reqId, code, msg, contract):
        if code in SUPPRESS_IB_CODES:
            return
        print(f"IB Error {code}, reqId {reqId}: {msg}")

    try:
        ib.errorEvent.clear()
    except Exception:
        pass
    ib.errorEvent += _quiet

    if delayed_only:
        # 3 = delayed streaming, 4 = delayed frozen
        for mkt_type in (3, 4):
            try:
                ib.reqMarketDataType(mkt_type)
                return
            except Exception:
                pass
    else:
        for mkt_type in (1, 3, 4):
            try:
                ib.reqMarketDataType(mkt_type)
                return
            except Exception:
                pass

def connect_ib(host="127.0.0.1", port=7497, client_id=None) -> IB:
    util.startLoop()
    ib = IB()

    ids = ([client_id] + [i for i in range(2, 199) if i != client_id]) if client_id is not None else list(range(2, 199))
    random.shuffle(ids)

    last = None
    for cid in ids:
        try:
            ib.connect(host, port, clientId=cid, readonly=True, timeout=10)
            if not ib.isConnected():
                raise TimeoutError("API connection failed")
            setup_marketdata_mode(ib, delayed_only=USE_DELAYED_ONLY)
            return ib
        except Exception as e:
            last = e
            try:
                ib.disconnect()
            except Exception:
                pass

    raise RuntimeError(f"Connessione IB fallita. Ultimo errore: {last}")

# =========================
# UNDERLYING
# =========================
def get_underlying_stock(ib: IB, ticker=TICKER, exchange=EXCHANGE, currency=CURRENCY):
    stk = Stock(ticker, exchange, currency)
    q = ib.qualifyContracts(stk)
    if not q:
        raise RuntimeError(f"Titolo {ticker} non qualificato.")
    return q[0]

def _pick_underlying_px_from_ticker(tk) -> Optional[float]:
    # per delayed spesso close o bid/ask sono più facili da ottenere
    b = getattr(tk, "bid", np.nan)
    a = getattr(tk, "ask", np.nan)
    last = getattr(tk, "last", np.nan)
    close = getattr(tk, "close", np.nan)

    if _finite_pos(b) and _finite_pos(a):
        return 0.5 * (float(b) + float(a))
    if _finite_pos(close):
        return float(close)
    if _finite_pos(last):
        return float(last)
    if _finite_pos(b):
        return float(b)
    if _finite_pos(a):
        return float(a)

    try:
        mp = tk.marketPrice()
        if _finite_pos(mp):
            return float(mp)
    except Exception:
        pass
    return None

def _hist_latest_bar(ib: IB, stk, days=7, bar="5 mins", what="TRADES", useRTH=False):
    try:
        bars: List[BarData] = ib.reqHistoricalData(
            stk,
            endDateTime="",
            durationStr=f"{days} D",
            barSizeSetting=bar,
            whatToShow=what,
            useRTH=useRTH,
            formatDate=1,
            keepUpToDate=False
        )
        if bars:
            last = bars[-1]
            return float(last.close), pd.Timestamp(last.date, tz="UTC")
    except Exception:
        pass
    return None, None

def _yesterday_close(ib: IB, stk):
    bars = ib.reqHistoricalData(
        stk,
        endDateTime="",
        durationStr="10 D",
        barSizeSetting="1 day",
        whatToShow="TRADES",
        useRTH=True,
        formatDate=1,
        keepUpToDate=False
    )
    if not bars:
        raise RuntimeError("Daily history insufficiente.")
    last = bars[-1]
    return float(last.close), pd.Timestamp(last.date, tz="UTC")

def fetch_underlying_ref(ib: IB, stk) -> Tuple[float, pd.Timestamp, str, float, pd.Timestamp]:
    # 1) delayed/live snapshot
    for wait in (2.5, 4.0):
        try:
            tk = ib.reqMktData(stk, "", snapshot=False, regulatorySnapshot=False)
            ib.sleep(wait)
            px = _pick_underlying_px_from_ticker(tk)
            try:
                ib.cancelMktData(stk)
            except Exception:
                pass
            if px is not None:
                ypx, yts = _yesterday_close(ib, stk)
                return float(px), pd.Timestamp.now("UTC"), "mktdata", ypx, yts
        except Exception:
            pass

    # 2) fallback storico intraday
    candidates = []
    for what in ("TRADES", "MIDPOINT", "BID", "ASK"):
        px, ts = _hist_latest_bar(ib, stk, days=7, bar="5 mins", what=what, useRTH=False)
        if px is not None and ts is not None:
            candidates.append((ts, px))
    if candidates:
        ts, px = max(candidates, key=lambda t: t[0])
        ypx, yts = _yesterday_close(ib, stk)
        return float(px), ts, "historical_intraday", ypx, yts

    # 3) daily fallback
    ypx, yts = _yesterday_close(ib, stk)
    return float(ypx), yts, "historical_daily", ypx, yts

# =========================
# OPTION CHAIN PARAMS
# =========================
def get_option_chain_for_stock(ib: IB, stk, max_years: float = 3.0):
    params = ib.reqSecDefOptParams(stk.symbol, "", stk.secType, stk.conId)
    if not params:
        raise RuntimeError(f"reqSecDefOptParams non ha restituito risultati per {stk.symbol} (conId={stk.conId}).")

    smart_candidates = [p for p in params if str(getattr(p, "exchange", "")).upper() == "SMART"]
    candidates = smart_candidates if smart_candidates else list(params)

    best = max(
        candidates,
        key=lambda p: (
            len(getattr(p, "expirations", []) or []),
            len(getattr(p, "strikes", []) or [])
        )
    )

    today_ny = _to_ny(pd.Timestamp.now("UTC")).normalize()
    cutoff = today_ny + pd.Timedelta(days=int(max_years * 365))

    expiries_ts = sorted(set(
        ts for s in (getattr(best, "expirations", []) or [])
        if (ts := _parse_expiry_to_ny_close(s)) is not None and today_ny <= ts <= cutoff
    ))
    if not expiries_ts:
        raise RuntimeError("Nessuna scadenza opzione disponibile dopo il filtro.")

    strikes = []
    for x in (getattr(best, "strikes", []) or []):
        try:
            fx = float(x)
            if math.isfinite(fx) and fx > 0:
                strikes.append(fx)
        except Exception:
            pass
    strikes = sorted(set(strikes))
    if not strikes:
        raise RuntimeError("Nessuno strike valido restituito da IB per la option chain.")

    return {
        "expiries_ts": expiries_ts,
        "strikes": strikes,
        "exchange": "SMART",
        "tradingClass": str(getattr(best, "tradingClass", "") or stk.symbol),
        "multiplier": str(getattr(best, "multiplier", "") or "100"),
    }

def select_expiries(all_exps: List[pd.Timestamp], mode: str = EXPIRY_MODE) -> List[pd.Timestamp]:
    if not all_exps:
        return []

    if mode == "all":
        return list(all_exps)

    now_ny = _to_ny(pd.Timestamp.now("UTC"))
    short = []
    long = []
    for x in all_exps:
        d = (x - now_ny).total_seconds() / (24 * 3600)
        if d <= SHORT_EXPIRY_DAYS_FULL:
            short.append(x)
        else:
            long.append(x)

    sampled_long = _sample_evenly(long, LONG_EXPIRY_SAMPLED)
    chosen = sorted(set(short + sampled_long))

    # assicurati di tenere la prima e l'ultima
    chosen = sorted(set(chosen + [all_exps[0], all_exps[-1]]))
    return chosen

def select_symmetric_strikes(chain_strikes: List[float], center: float) -> Tuple[float, ...]:
    if not chain_strikes:
        return tuple()

    below = sorted([k for k in chain_strikes if k < center])
    above = sorted([k for k in chain_strikes if k > center])

    atm = min(chain_strikes, key=lambda x: abs(x - center))

    below_near = below[-NEAR_STRIKES_EACH_SIDE:] if below else []
    above_near = above[:NEAR_STRIKES_EACH_SIDE] if above else []

    below_far_pool = below[:-NEAR_STRIKES_EACH_SIDE] if len(below) > NEAR_STRIKES_EACH_SIDE else []
    above_far_pool = above[NEAR_STRIKES_EACH_SIDE:] if len(above) > NEAR_STRIKES_EACH_SIDE else []

    below_far = _sample_evenly(below_far_pool, FAR_STRIKES_EACH_SIDE)
    above_far = _sample_evenly(above_far_pool, FAR_STRIKES_EACH_SIDE)

    out = set(below_near + above_near + below_far + above_far + [atm])

    if INCLUDE_EXTREME_STRIKES:
        out.add(chain_strikes[0])
        out.add(chain_strikes[-1])

    return tuple(sorted(float(x) for x in out))

# =========================
# QUOTE EXTRACTION
# =========================
def _get_model_greek_field(tk, field: str):
    try:
        mg = tk.modelGreeks
        if mg is None:
            return np.nan
        return _f(getattr(mg, field, np.nan))
    except Exception:
        return np.nan

def _get_option_metrics(tk):
    bid = _f(getattr(tk, "bid", np.nan))
    ask = _f(getattr(tk, "ask", np.nan))
    last = _f(getattr(tk, "last", np.nan))
    close = _f(getattr(tk, "close", np.nan))

    mid = np.nan
    if _finite_pos(bid) and _finite_pos(ask):
        mid = 0.5 * (bid + ask)

    model_price = _get_model_greek_field(tk, "optPrice")
    iv = _get_model_greek_field(tk, "impliedVol")
    delta = _get_model_greek_field(tk, "delta")
    gamma = _get_model_greek_field(tk, "gamma")
    vega = _get_model_greek_field(tk, "vega")
    theta = _get_model_greek_field(tk, "theta")
    und_price = _get_model_greek_field(tk, "undPrice")

    volume = _f(getattr(tk, "volume", np.nan))
    open_interest = _f(getattr(tk, "putOpenInterest", np.nan))
    if not _finite(open_interest):
        open_interest = _f(getattr(tk, "callOpenInterest", np.nan))

    # ordine di preferenza per delayed/no-subscription:
    # mid -> close -> last -> model -> bid -> ask
    chosen_price = np.nan
    chosen_source = None
    for name, val in [
        ("mid", mid),
        ("close", close),
        ("last", last),
        ("model", model_price),
        ("bid", bid),
        ("ask", ask),
    ]:
        if _finite_pos(val):
            chosen_price = float(val)
            chosen_source = name
            break

    has_any_quote = any(_finite_pos(x) for x in [bid, ask, mid, last, close, model_price])

    return {
        "bid": bid,
        "ask": ask,
        "mid": mid,
        "last": last,
        "close": close,
        "model_price": model_price,
        "price_used": chosen_price,
        "price_source": chosen_source,
        "has_any_quote": bool(has_any_quote),
        "impliedVol": iv,
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "theta": theta,
        "undPrice_model": und_price,
        "volume": volume,
        "openInterest": open_interest,
    }

def fetch_chain_selected(
    ib: IB,
    stk,
    expiries_ts: List[pd.Timestamp],
    option_exchange: str,
    tradingClass: str,
    multiplier: str,
    strikes: Tuple[float, ...],
    include_calls=True,
    include_puts=True,
    sleep_seconds=SLEEP_SECONDS,
    passes_per_batch=PASSES_PER_BATCH,
    batch_size=BATCH_SIZE,
    generic_ticks=GENERIC_TICKS
) -> pd.DataFrame:
    S0, spot_ts, source, y_close, y_close_ts = fetch_underlying_ref(ib, stk)
    now_ny = _to_ny(pd.Timestamp.now("UTC"))

    rights = []
    if include_calls:
        rights.append("C")
    if include_puts:
        rights.append("P")

    rows = []
    report = []

    for exp_ts in expiries_ts:
        expiry_ymd = exp_ts.strftime("%Y%m%d")
        T = yearfrac_365(now_ny, exp_ts)
        if T <= 0:
            report.append((expiry_ymd, 0, "T<=0"))
            continue

        contracts = [
            Option(
                symbol=stk.symbol,
                lastTradeDateOrContractMonth=expiry_ymd,
                strike=float(k),
                right=r,
                exchange=option_exchange,
                currency=stk.currency,
                multiplier=multiplier,
                tradingClass=tradingClass
            )
            for r in rights for k in strikes
        ]

        qualified = ib.qualifyContracts(*contracts)
        report.append((expiry_ymd, len(qualified), f"qualified_from_requested={len(contracts)}"))

        if not qualified:
            continue

        seen_conids = set()

        for _ in range(passes_per_batch):
            for i in range(0, len(qualified), batch_size):
                qs = qualified[i:i + batch_size]
                tks = [ib.reqMktData(c, generic_ticks, snapshot=False, regulatorySnapshot=False) for c in qs]
                ib.sleep(sleep_seconds)

                for c, tk in zip(qs, tks):
                    if c.conId in seen_conids:
                        continue

                    metrics = _get_option_metrics(tk)
                    rows.append({
                        "expiry_iso": str(exp_ts.isoformat()),
                        "expiry_ymd": expiry_ymd,
                        "T": T,
                        "right": c.right,
                        "K": float(c.strike),
                        "conId": c.conId,
                        "localSymbol": c.localSymbol,
                        "symbol": c.symbol,
                        **metrics
                    })
                    seen_conids.add(c.conId)

                for c in qs:
                    try:
                        ib.cancelMktData(c)
                    except Exception:
                        pass

                time.sleep(PAUSE_BETWEEN_BATCHES)

    if not rows:
        raise RuntimeError("Nessun contratto/opzione raccolto.")

    df = pd.DataFrame(rows).sort_values(
        ["expiry_ymd", "right", "K"]
    ).reset_index(drop=True)

    df.attrs["S0"] = S0
    df.attrs["spot_ts"] = spot_ts
    df.attrs["source"] = source
    df.attrs["y_close"] = y_close
    df.attrs["y_close_ts"] = y_close_ts
    df.attrs["fetch_report"] = report
    return df

# =========================
# PIPELINE
# =========================
def run_pipeline(
    ticker=TICKER,
    exchange=EXCHANGE,
    currency=CURRENCY,
    dates_it=DATES_IT,
    out_prefix=OUT_PREFIX
):
    ib = connect_ib()
    try:
        stk = get_underlying_stock(ib, ticker=ticker, exchange=exchange, currency=currency)

        chain = get_option_chain_for_stock(ib, stk)
        all_exps = chain["expiries_ts"]
        tradingClass = chain["tradingClass"]
        multiplier = chain["multiplier"]
        option_exchange = chain["exchange"]
        chain_strikes = chain["strikes"]

        if USE_ALL_EXPIRIES:
            exps = select_expiries(all_exps, mode=EXPIRY_MODE)
        else:
            exps = parse_user_expiries_it(dates_it)
            known = {x.strftime("%Y%m%d") for x in all_exps}
            exps = [x for x in exps if x.strftime("%Y%m%d") in known]

        if not exps:
            raise RuntimeError("Lista scadenze vuota/non valida dopo il filtro.")

        S0_guess, _, _, _, _ = fetch_underlying_ref(ib, stk)
        strikes = select_symmetric_strikes(chain_strikes, center=S0_guess)
        if not strikes:
            raise RuntimeError("Nessuno strike selezionato.")

        df = fetch_chain_selected(
            ib=ib,
            stk=stk,
            expiries_ts=exps,
            option_exchange=option_exchange,
            tradingClass=tradingClass,
            multiplier=multiplier,
            strikes=strikes,
            include_calls=True,
            include_puts=True
        )

        as_of_ny = str(_to_ny(pd.Timestamp.now("UTC")).normalize().date())

        meta = {
            "underlying_symbol": ticker,
            "underlying_exchange": exchange,
            "underlying_currency": currency,
            "market_data_mode": "delayed_only" if USE_DELAYED_ONLY else "live_or_delayed",
            "option_exchange": option_exchange,
            "option_tradingClass": tradingClass,
            "option_multiplier": multiplier,
            "S0": float(df.attrs["S0"]),
            "spot_ts": str(df.attrs["spot_ts"]),
            "as_of_ny": as_of_ny,
            "source": df.attrs["source"],
            "y_close": float(df.attrs["y_close"]),
            "y_close_ts": str(df.attrs["y_close_ts"]),
            "n_expiries_selected": len(exps),
            "n_strikes_selected": len(strikes),
            "strike_min": float(min(strikes)),
            "strike_max": float(max(strikes)),
            "n_rows_total": int(len(df)),
            "n_rows_with_any_quote": int(df["has_any_quote"].fillna(False).sum()),
            "n_rows_with_price_used": int(df["price_used"].notna().sum()),
            "expiry_mode": EXPIRY_MODE,
            "short_expiry_days_full": SHORT_EXPIRY_DAYS_FULL,
            "long_expiry_sampled": LONG_EXPIRY_SAMPLED,
            "near_strikes_each_side": NEAR_STRIKES_EACH_SIDE,
            "far_strikes_each_side": FAR_STRIKES_EACH_SIDE,
            "include_extreme_strikes": INCLUDE_EXTREME_STRIKES,
            "fetch_report": df.attrs["fetch_report"],
        }

        os.makedirs(DATA_DIR, exist_ok=True)

        csv_path = os.path.join(DATA_DIR, f"{out_prefix}_chain.csv")
        json_path = os.path.join(DATA_DIR, f"{out_prefix}_meta.json")

        df.to_csv(csv_path, index=False)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        print(f"[FETCH] rows={len(df)} -> {csv_path} / {json_path}")
        print(f"[INFO] Underlying={ticker}  S0={meta['S0']:.4f}  source={meta['source']}  mode={meta['market_data_mode']}")
        print(f"[INFO] Expiries selected={meta['n_expiries_selected']}  Strikes selected={meta['n_strikes_selected']}")
        print(f"[INFO] Rows with any quote={meta['n_rows_with_any_quote']} / {meta['n_rows_total']}")
        print(f"[INFO] Rows with price_used={meta['n_rows_with_price_used']} / {meta['n_rows_total']}")
        print("[NOTE] Il CSV non elimina i contratti senza prezzo: li salva lo stesso.")
        print("[NOTE] Usa 'price_used' come prezzo sintetico, ma conserva anche bid/ask/mid/last/close/model_price.")

    finally:
        try:
            ib.disconnect()
        except Exception:
            pass

if __name__ == "__main__":
    run_pipeline()

## Data cleaning

In [ ]:
import os
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
FILE_PATH = "data/gld_chain_wide_chain.csv"
OUTPUT_PATH = "data/gld_chain_clean.csv"

TARGET_POINTS = 100
MIN_PRICE = 1.0   # requisito: prezzo strettamente maggiore di 1
MIN_GRID_SIZE = 4 # minimo numero di bin per asse nella griglia

# =========================
# CHECK CURVA NSS
# =========================
# Assumo che tu abbia già eseguito:
#
# yield_maturities = np.array([1/12, 1.5/12, 2/12, 3/12, 4/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30])
# yields = np.array([3.72,3.72,3.73,3.70,3.71,3.72,3.68,3.79,3.82,3.94,4.12,4.31,4.88,4.88]).astype(float) / 100
# curve_fit, status = calibrate_nss_ols(yield_maturities, yields)

if "curve_fit" not in globals():
    raise RuntimeError("curve_fit non esiste. Esegui prima calibrate_nss_ols(...).")

if not callable(curve_fit):
    raise RuntimeError("curve_fit esiste ma non è callable.")

# =========================
# FUNZIONI UTILI
# =========================
def normalize_columns(df, cols):
    out = df.copy()
    for c in cols:
        cmin = out[c].min()
        cmax = out[c].max()
        span = cmax - cmin
        if span <= 0:
            out[c + "_norm"] = 0.0
        else:
            out[c + "_norm"] = (out[c] - cmin) / span
    return out

def farthest_point_fill(candidates, n_to_add, seed=None):
    """
    Sceglie n_to_add punti dai candidates massimizzando la dispersione in (T, K).
    Se seed è fornito, i nuovi punti vengono scelti anche lontani dai seed.
    """
    if n_to_add <= 0 or candidates.empty:
        return candidates.iloc[0:0].copy()

    candidates = candidates.copy().reset_index(drop=True)

    all_parts = [candidates[["T", "K"]].copy()]
    if seed is not None and not seed.empty:
        all_parts.append(seed[["T", "K"]].copy())

    ref = pd.concat(all_parts, ignore_index=True)
    t_min, t_max = ref["T"].min(), ref["T"].max()
    k_min, k_max = ref["K"].min(), ref["K"].max()

    t_span = max(t_max - t_min, 1e-12)
    k_span = max(k_max - k_min, 1e-12)

    cand_pts = np.column_stack([
        (candidates["T"].values - t_min) / t_span,
        (candidates["K"].values - k_min) / k_span
    ])

    if seed is not None and not seed.empty:
        seed_pts = np.column_stack([
            (seed["T"].values - t_min) / t_span,
            (seed["K"].values - k_min) / k_span
        ])

        min_dist2 = np.full(len(candidates), np.inf)
        for s in seed_pts:
            d2 = np.sum((cand_pts - s) ** 2, axis=1)
            min_dist2 = np.minimum(min_dist2, d2)
    else:
        center = np.array([0.5, 0.5])
        first_idx = np.argmin(np.sum((cand_pts - center) ** 2, axis=1))
        selected = [first_idx]

        min_dist2 = np.sum((cand_pts - cand_pts[first_idx]) ** 2, axis=1)

        while len(selected) < min(n_to_add, len(candidates)):
            next_idx = int(np.argmax(min_dist2))
            if next_idx in selected:
                break
            selected.append(next_idx)
            d2 = np.sum((cand_pts - cand_pts[next_idx]) ** 2, axis=1)
            min_dist2 = np.minimum(min_dist2, d2)

        return candidates.iloc[selected].copy()

    selected = []
    current_min_dist2 = min_dist2.copy()

    while len(selected) < min(n_to_add, len(candidates)):
        next_idx = int(np.argmax(current_min_dist2))
        if next_idx in selected:
            break

        selected.append(next_idx)

        d2 = np.sum((cand_pts - cand_pts[next_idx]) ** 2, axis=1)
        current_min_dist2 = np.minimum(current_min_dist2, d2)
        current_min_dist2[selected] = -1.0

    return candidates.iloc[selected].copy()

def select_uniform_nodes(surface_df, target_points, min_grid_size=4):
    """
    Selezione uniforme nel piano (T, K):
    1) griglia regolare su T e K
    2) prende al più un punto per cella, quello più vicino al centro della cella
    3) se mancano punti rispetto a target_points, li aggiunge con farthest point fill
    """
    if surface_df.empty:
        return surface_df.copy()

    data = surface_df.copy().reset_index(drop=True)
    data["row_id"] = np.arange(len(data))

    n_target = min(target_points, len(data))

    # dimensione della griglia circa quadrata
    n_t_bins = max(min_grid_size, int(round(np.sqrt(n_target))))
    n_k_bins = max(min_grid_size, int(np.ceil(n_target / n_t_bins)))

    t_min, t_max = data["T"].min(), data["T"].max()
    k_min, k_max = data["K"].min(), data["K"].max()

    if t_max <= t_min or k_max <= k_min:
        # fallback: se T o K non variano abbastanza
        return farthest_point_fill(data, n_target).sort_values(["T", "K"]).reset_index(drop=True)

    t_edges = np.linspace(t_min, t_max, n_t_bins + 1)
    k_edges = np.linspace(k_min, k_max, n_k_bins + 1)

    chosen_rows = []

    for i in range(n_t_bins):
        t_lo, t_hi = t_edges[i], t_edges[i + 1]

        if i < n_t_bins - 1:
            mask_t = (data["T"] >= t_lo) & (data["T"] < t_hi)
        else:
            mask_t = (data["T"] >= t_lo) & (data["T"] <= t_hi)

        for j in range(n_k_bins):
            k_lo, k_hi = k_edges[j], k_edges[j + 1]

            if j < n_k_bins - 1:
                mask_k = (data["K"] >= k_lo) & (data["K"] < k_hi)
            else:
                mask_k = (data["K"] >= k_lo) & (data["K"] <= k_hi)

            cell = data[mask_t & mask_k].copy()
            if cell.empty:
                continue

            t_center = 0.5 * (t_lo + t_hi)
            k_center = 0.5 * (k_lo + k_hi)

            dist2 = ((cell["T"] - t_center) / (t_max - t_min)) ** 2 + ((cell["K"] - k_center) / (k_max - k_min)) ** 2
            chosen_rows.append(cell.loc[dist2.idxmin()])

    if len(chosen_rows) == 0:
        # fallback robusto
        selected = farthest_point_fill(data, n_target)
        return selected.sort_values(["T", "K"]).reset_index(drop=True)

    selected = pd.DataFrame(chosen_rows).drop_duplicates(subset=["row_id"]).copy()

    # Se abbiamo troppi punti, riduciamo con farthest point sampling
    if len(selected) > n_target:
        selected = farthest_point_fill(selected, n_target)

    # Se abbiamo troppo pochi punti, riempiamo con punti lontani dai già scelti
    elif len(selected) < n_target:
        remaining = data[~data["row_id"].isin(selected["row_id"])].copy()
        needed = n_target - len(selected)
        extra = farthest_point_fill(remaining, needed, seed=selected)
        selected = pd.concat([selected, extra], ignore_index=True)

    selected = selected.drop_duplicates(subset=["row_id"]).copy()
    selected = selected.sort_values(["T", "K"]).reset_index(drop=True)

    # sicurezza finale
    if len(selected) > n_target:
        selected = farthest_point_fill(selected, n_target).sort_values(["T", "K"]).reset_index(drop=True)

    return selected

# =========================
# LETTURA FILE
# =========================
df = pd.read_csv(FILE_PATH)

# =========================
# CONTROLLI COLONNE
# =========================
required_cols = ["right", "price_used", "K", "T"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise RuntimeError(f"Mancano colonne necessarie: {missing}")

# =========================
# PARSE SCADENZA
# =========================
if "expiry_ymd" in df.columns:
    df["expiry"] = pd.to_datetime(
        df["expiry_ymd"].astype(str),
        format="%Y%m%d",
        errors="coerce"
    ).dt.date
elif "expiry_iso" in df.columns:
    tmp = pd.to_datetime(df["expiry_iso"], errors="coerce", utc=True)
    df["expiry"] = tmp.dt.date
else:
    raise RuntimeError("Non trovo né 'expiry_ymd' né 'expiry_iso' nel file.")

# =========================
# PULIZIA BASE
# =========================
df = df[df["right"] == "C"].copy()

df["price_used"] = pd.to_numeric(df["price_used"], errors="coerce")
df["K"] = pd.to_numeric(df["K"], errors="coerce")
df["T"] = pd.to_numeric(df["T"], errors="coerce")

df = df.dropna(subset=["expiry", "T", "K", "price_used"]).copy()
df = df[df["price_used"] > MIN_PRICE].copy()
df = df[df["K"] > 0].copy()
df = df[df["T"] > 0].copy()

# duplicati stessa scadenza/strike
df = (
    df.sort_values(["expiry", "K"])
      .drop_duplicates(subset=["expiry", "K"], keep="first")
      .copy()
)

surface_df = df[["expiry", "T", "K", "price_used"]].rename(
    columns={"price_used": "price"}
).copy()

surface_df = surface_df.sort_values(["T", "K"]).reset_index(drop=True)

if surface_df.empty:
    raise RuntimeError("Dopo la pulizia non sono rimasti dati validi.")

# =========================
# SELEZIONE NODI PIU' UNIFORME NEL PIANO (T, K)
# =========================
calib_df = select_uniform_nodes(
    surface_df=surface_df,
    target_points=TARGET_POINTS,
    min_grid_size=MIN_GRID_SIZE
).copy()

if calib_df.empty:
    raise RuntimeError("Il dataset finale per la calibrazione è vuoto.")

# =========================
# TASSO DA NELSON-SIEGEL-SVENSSON
# =========================
calib_df["rate"] = np.asarray(curve_fit(calib_df["T"].values), dtype=float)
calib_df["discount_factor"] = np.exp(-calib_df["rate"] * calib_df["T"])

# =========================
# COLONNE FINALI
# =========================
calib_df = calib_df[["expiry", "T", "K", "price", "rate", "discount_factor"]].copy()
calib_df = calib_df.sort_values(["T", "K"]).reset_index(drop=True)

# =========================
# SALVATAGGIO
# =========================
os.makedirs("Data", exist_ok=True)
calib_df.to_csv(OUTPUT_PATH, index=False)

# =========================
# OUTPUT
# =========================
print("Numero punti finali:", len(calib_df))
print("Numero scadenze finali:", calib_df["expiry"].nunique())
print("Prezzo minimo richiesto:", MIN_PRICE)
print("File salvato in:", OUTPUT_PATH)
print()
print(calib_df.head(30))

try:
    from IPython.display import display
    display(calib_df)
except Exception:
    pass

## surface

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

# forza apertura nel browser
pio.renderers.default = "browser"

# dati
x = calib_df["T"].values
y = calib_df["K"].values
z = calib_df["price"].values

fig = go.Figure()

fig.add_trace(
    go.Mesh3d(
        x=x,
        y=y,
        z=z,
        intensity=z,
        opacity=0.85,
        name="Surface"
    )
)

fig.add_trace(
    go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="markers",
        marker=dict(size=3),
        name="Observed points"
    )
)

fig.update_layout(
    title="GLD Call Price Surface",
    scene=dict(
        xaxis_title="T",
        yaxis_title="Strike K",
        zaxis_title="Option Price"
    ),
    width=950,
    height=750
)

fig.show()

In [ ]:
import json
import os
from math import erf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======================================================
# CONFIG
# ======================================================
META_JSON_PATH = "data/gld_chain_wide_meta.json"
DIVIDEND_YIELD = 0.0   # se non hai q, lascia 0
MIN_VOL = 1e-8
MAX_VOL = 5.0
VOL_TOL = 1e-8
MAX_ITERS = 200
OUTPUT_IV_PATH = "data/gld_chain_with_iv.csv"

# ======================================================
# CHECK INPUT
# ======================================================
if "calib_df" not in globals():
    raise RuntimeError("calib_df non esiste. Esegui prima il codice che costruisce calib_df.")

required_cols = ["expiry", "T", "K", "price", "rate"]
missing = [c for c in required_cols if c not in calib_df.columns]
if missing:
    raise RuntimeError(f"In calib_df mancano colonne necessarie: {missing}")

# ======================================================
# LETTURA SPOT DAL JSON
# ======================================================
def find_key_recursive(obj, candidate_keys):
    """
    Cerca ricorsivamente la prima chiave presente in candidate_keys
    dentro dict/list annidati.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in candidate_keys:
                return v
        for v in obj.values():
            out = find_key_recursive(v, candidate_keys)
            if out is not None:
                return out

    elif isinstance(obj, list):
        for item in obj:
            out = find_key_recursive(item, candidate_keys)
            if out is not None:
                return out

    return None


def load_spot_from_json(json_path):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"File JSON non trovato: {json_path}")

    with open(json_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    # chiavi candidate
    candidate_keys = ["S0", "spot", "spot_price", "underlying_price", "underlying_last"]

    raw_spot = find_key_recursive(meta, candidate_keys)

    if raw_spot is None:
        raise RuntimeError(
            f"Non trovo nessuna chiave spot nel JSON. "
            f"Chiavi cercate: {candidate_keys}"
        )

    try:
        spot = float(raw_spot)
    except Exception:
        raise RuntimeError(f"Valore spot trovato ma non convertibile a float: {raw_spot}")

    if not np.isfinite(spot) or spot <= 0:
        raise RuntimeError(f"Spot non valido letto dal JSON: {spot}")

    return spot, meta


S0, meta_json = load_spot_from_json(META_JSON_PATH)
print(f"Spot letto automaticamente dal JSON: S0 = {S0:.6f}")

# ======================================================
# FUNZIONI BLACK-SCHOLES
# ======================================================
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))


def bs_call_price(S, K, T, r, sigma, q=0.0):
    if T <= 0:
        return max(S - K, 0.0)

    if sigma <= 0:
        return max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)

    sqrtT = np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    return (
        S * np.exp(-q * T) * norm_cdf(d1)
        - K * np.exp(-r * T) * norm_cdf(d2)
    )


def implied_vol_call(price, S, K, T, r, q=0.0,
                     vol_low=MIN_VOL, vol_high=MAX_VOL,
                     tol=VOL_TOL, max_iters=MAX_ITERS):
    """
    Inversione numerica Black-Scholes per call europea via bisezione.
    Restituisce np.nan se il prezzo non è ammissibile.
    """

    # controlli base
    vals = [price, S, K, T, r]
    if not all(np.isfinite(v) for v in vals):
        return np.nan

    if S <= 0 or K <= 0 or T <= 0 or price <= 0:
        return np.nan

    discounted_spot = S * np.exp(-q * T)
    discounted_strike = K * np.exp(-r * T)

    lower_bound = max(discounted_spot - discounted_strike, 0.0)
    upper_bound = discounted_spot

    eps = 1e-10
    if price < lower_bound - eps or price > upper_bound + eps:
        return np.nan

    if abs(price - lower_bound) < 1e-12:
        return 0.0

    f_low = bs_call_price(S, K, T, r, vol_low, q) - price
    f_high = bs_call_price(S, K, T, r, vol_high, q) - price

    # allargo il bound superiore se necessario
    current_high = vol_high
    for _ in range(12):
        if f_low * f_high <= 0:
            break
        current_high *= 2.0
        f_high = bs_call_price(S, K, T, r, current_high, q) - price
    else:
        return np.nan

    a = vol_low
    b = current_high
    fa = f_low
    fb = f_high

    for _ in range(max_iters):
        m = 0.5 * (a + b)
        fm = bs_call_price(S, K, T, r, m, q) - price

        if abs(fm) < tol or (b - a) < tol:
            return m

        if fa * fm <= 0:
            b = m
            fb = fm
        else:
            a = m
            fa = fm

    return 0.5 * (a + b)

# ======================================================
# CALCOLO IMPLIED VOL
# ======================================================
iv_df = calib_df.copy()

iv_df["implied_vol"] = iv_df.apply(
    lambda row: implied_vol_call(
        price=float(row["price"]),
        S=S0,
        K=float(row["K"]),
        T=float(row["T"]),
        r=float(row["rate"]),
        q=DIVIDEND_YIELD
    ),
    axis=1
)

iv_df = iv_df[np.isfinite(iv_df["implied_vol"])].copy()
iv_df = iv_df[iv_df["implied_vol"] >= 0].copy()
iv_df["implied_vol_pct"] = 100.0 * iv_df["implied_vol"]

if iv_df.empty:
    raise RuntimeError("Nessuna volatilità implicita valida calcolata.")

iv_df = iv_df.sort_values(["expiry", "K"]).reset_index(drop=True)

# ======================================================
# SALVATAGGIO
# ======================================================
os.makedirs("Data", exist_ok=True)
iv_df.to_csv(OUTPUT_IV_PATH, index=False)

# ======================================================
# GRAFICO 2D: CURVE IV PER SCADENZA
# ======================================================
plt.figure(figsize=(13, 8))

for expiry, g in iv_df.groupby("expiry", sort=True):
    g = g.sort_values("K")
    plt.plot(
        g["K"].values,
        g["implied_vol_pct"].values,
        marker="o",
        linewidth=1.5,
        markersize=3.5,
        label=str(expiry)
    )

plt.xlabel("Strike K")
plt.ylabel("Implied Volatility (%)")
plt.title("GLD - Implied Volatility Curves by Expiry")
plt.grid(True, alpha=0.3)
plt.legend(
    title="Expiry",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
    ncol=1
)
plt.tight_layout()
plt.show()

# ======================================================
# OUTPUT
# ======================================================
print("Numero punti con IV valida:", len(iv_df))
print("Numero scadenze nel grafico:", iv_df["expiry"].nunique())
print("Spot usato:", S0)
print("File salvato in:", OUTPUT_IV_PATH)
print()
print(iv_df[["expiry", "T", "K", "price", "rate", "implied_vol", "implied_vol_pct"]].head(30))

try:
    from IPython.display import display
    display(iv_df[["expiry", "T", "K", "price", "rate", "implied_vol", "implied_vol_pct"]])
except Exception:
    pass

In [ ]:
import json
import os
from math import erf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======================================================
# CONFIG
# ======================================================
META_JSON_PATH = "data/gld_chain_wide_meta.json"

K_MIN = 200.0
K_MAX = 600.0

DIVIDEND_YIELD = 0.0
MIN_VOL = 1e-8
MAX_VOL = 5.0
VOL_TOL = 1e-8
MAX_ITERS = 200

# tengo solo scadenze con almeno 5 punti validi nel range strike scelto
MIN_POINTS_TO_KEEP_EXPIRY = 5

OUTPUT_DATASET_PATH = "data/gld_iv_dataset_200_600.pkl"
OUTPUT_IMAGE_PATH = "data/gld_iv_curves_200_600.png"

# ======================================================
# CHECK INPUT
# ======================================================
if "calib_df" not in globals():
    raise RuntimeError("calib_df non esiste. Esegui prima il codice che costruisce calib_df.")

required_cols = ["expiry", "T", "K", "price", "rate"]
missing = [c for c in required_cols if c not in calib_df.columns]
if missing:
    raise RuntimeError(f"In calib_df mancano colonne necessarie: {missing}")

# ======================================================
# LETTURA SPOT DAL JSON
# ======================================================
def find_key_recursive(obj, candidate_keys):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in candidate_keys:
                return v
        for v in obj.values():
            out = find_key_recursive(v, candidate_keys)
            if out is not None:
                return out
    elif isinstance(obj, list):
        for item in obj:
            out = find_key_recursive(item, candidate_keys)
            if out is not None:
                return out
    return None

def load_spot_from_json(json_path):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"File JSON non trovato: {json_path}")

    with open(json_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    candidate_keys = ["S0", "spot", "spot_price", "underlying_price", "underlying_last"]
    raw_spot = find_key_recursive(meta, candidate_keys)

    if raw_spot is None:
        raise RuntimeError(f"Non trovo lo spot nel JSON. Chiavi cercate: {candidate_keys}")

    spot = float(raw_spot)

    if not np.isfinite(spot) or spot <= 0:
        raise RuntimeError(f"Spot non valido letto dal JSON: {spot}")

    return spot, meta

S0, meta_json = load_spot_from_json(META_JSON_PATH)
print(f"Spot letto automaticamente dal JSON: S0 = {S0:.6f}")

# ======================================================
# FUNZIONI BLACK-SCHOLES
# ======================================================
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

def bs_call_price(S, K, T, r, sigma, q=0.0):
    if T <= 0:
        return max(S - K, 0.0)

    if sigma <= 0:
        return max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)

    sqrtT = np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    return (
        S * np.exp(-q * T) * norm_cdf(d1)
        - K * np.exp(-r * T) * norm_cdf(d2)
    )

def implied_vol_call(price, S, K, T, r, q=0.0,
                     vol_low=MIN_VOL, vol_high=MAX_VOL,
                     tol=VOL_TOL, max_iters=MAX_ITERS):
    vals = [price, S, K, T, r]
    if not all(np.isfinite(v) for v in vals):
        return np.nan

    if S <= 0 or K <= 0 or T <= 0 or price <= 0:
        return np.nan

    discounted_spot = S * np.exp(-q * T)
    discounted_strike = K * np.exp(-r * T)

    lower_bound = max(discounted_spot - discounted_strike, 0.0)
    upper_bound = discounted_spot

    eps = 1e-10
    if price < lower_bound - eps or price > upper_bound + eps:
        return np.nan

    if abs(price - lower_bound) < 1e-12:
        return 0.0

    f_low = bs_call_price(S, K, T, r, vol_low, q) - price
    f_high = bs_call_price(S, K, T, r, vol_high, q) - price

    current_high = vol_high
    for _ in range(12):
        if f_low * f_high <= 0:
            break
        current_high *= 2.0
        f_high = bs_call_price(S, K, T, r, current_high, q) - price
    else:
        return np.nan

    a = vol_low
    b = current_high
    fa = f_low

    for _ in range(max_iters):
        m = 0.5 * (a + b)
        fm = bs_call_price(S, K, T, r, m, q) - price

        if abs(fm) < tol or (b - a) < tol:
            return m

        if fa * fm <= 0:
            b = m
        else:
            a = m
            fa = fm

    return 0.5 * (a + b)

# ======================================================
# CAMPIONAMENTO UNIFORME CON STESSO NUMERO DI PUNTI
# ======================================================
def sample_evenly_same_count(group, n_points):
    group = group.sort_values("K").reset_index(drop=True)

    if len(group) < n_points:
        raise RuntimeError(
            f"Il gruppo con expiry={group['expiry'].iloc[0]} ha solo {len(group)} punti, "
            f"meno dei {n_points} richiesti."
        )

    if len(group) == n_points:
        return group.copy()

    idx = np.linspace(0, len(group) - 1, n_points).round().astype(int)
    idx = np.unique(idx)

    while len(idx) < n_points:
        extra = np.setdiff1d(np.arange(len(group)), idx)
        if len(extra) == 0:
            break
        idx = np.sort(np.concatenate([idx, [extra[0]]]))

    idx = idx[:n_points]
    return group.iloc[idx].copy()

# ======================================================
# COSTRUZIONE DATASET IV
# ======================================================
iv_df = calib_df.copy()

# filtro strike
iv_df = iv_df[(iv_df["K"] >= K_MIN) & (iv_df["K"] <= K_MAX)].copy()

if iv_df.empty:
    raise RuntimeError("Dopo il filtro strike [200, 600] non è rimasto nulla.")

# calcolo IV
iv_df["implied_vol"] = iv_df.apply(
    lambda row: implied_vol_call(
        price=float(row["price"]),
        S=S0,
        K=float(row["K"]),
        T=float(row["T"]),
        r=float(row["rate"]),
        q=DIVIDEND_YIELD
    ),
    axis=1
)

# tengo solo IV valide
iv_df = iv_df[np.isfinite(iv_df["implied_vol"])].copy()
iv_df = iv_df[iv_df["implied_vol"] > 0].copy()
iv_df = iv_df.sort_values(["expiry", "K"]).reset_index(drop=True)

if iv_df.empty:
    raise RuntimeError("Nessuna volatilità implicita valida dopo il filtro strike.")

# ======================================================
# ELIMINA LE SCADENZE CON MENO DI 5 PUNTI VALIDI
# ======================================================
counts_before = iv_df.groupby("expiry").size().sort_index()

valid_expiries = counts_before[counts_before >= MIN_POINTS_TO_KEEP_EXPIRY].index
dropped_expiries = counts_before[counts_before < MIN_POINTS_TO_KEEP_EXPIRY].index

iv_df = iv_df[iv_df["expiry"].isin(valid_expiries)].copy()

if iv_df.empty:
    raise RuntimeError(
        "Dopo aver eliminato le scadenze con meno di 5 punti validi non è rimasto nulla."
    )

# ======================================================
# STESSO NUMERO DI PUNTI PER OGNI SCADENZA RIMASTA
# ======================================================
counts_after = iv_df.groupby("expiry").size().sort_index()
points_per_expiry = int(counts_after.min())

if points_per_expiry < MIN_POINTS_TO_KEEP_EXPIRY:
    raise RuntimeError("Errore interno: il numero minimo di punti per scadenza è sceso sotto 5.")

uniform_groups = []
for expiry, g in iv_df.groupby("expiry", sort=True):
    uniform_groups.append(sample_evenly_same_count(g, points_per_expiry))

iv_uniform_df = pd.concat(uniform_groups, ignore_index=True)
iv_uniform_df = iv_uniform_df.sort_values(["expiry", "K"]).reset_index(drop=True)
iv_uniform_df["implied_vol_pct"] = 100.0 * iv_uniform_df["implied_vol"]

# ======================================================
# SALVATAGGIO DATASET
# ======================================================
os.makedirs("Data", exist_ok=True)
iv_uniform_df.to_pickle(OUTPUT_DATASET_PATH)

# ======================================================
# GRAFICO
# ======================================================
plt.figure(figsize=(13, 8))

for expiry, g in iv_uniform_df.groupby("expiry", sort=True):
    g = g.sort_values("K")
    plt.plot(
        g["K"].values,
        g["implied_vol_pct"].values,
        marker="o",
        linewidth=1.6,
        markersize=4,
        label=str(expiry)
    )

plt.xlabel("Strike K")
plt.ylabel("Implied Volatility (%)")
plt.title("GLD - Implied Volatility Curves by Expiry (K in [200, 600])")
plt.grid(True, alpha=0.3)
plt.legend(title="Expiry", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()

plt.savefig(OUTPUT_IMAGE_PATH, dpi=200, bbox_inches="tight")
plt.show()

# ======================================================
# OUTPUT
# ======================================================
print("Spot usato:", S0)
print("Range strike usato:", (K_MIN, K_MAX))
print("Numero scadenze finali:", iv_uniform_df["expiry"].nunique())
print("Punti per ciascuna scadenza:", points_per_expiry)
print("Numero totale righe dataset finale:", len(iv_uniform_df))
print("Dataset salvato in:", OUTPUT_DATASET_PATH)
print("Immagine salvata in:", OUTPUT_IMAGE_PATH)

if len(dropped_expiries) > 0:
    print()
    print("Scadenze escluse perché con meno di 5 punti validi:")
    for exp in dropped_expiries:
        print(" -", exp)

print()
print(iv_uniform_df[["expiry", "T", "K", "price", "rate", "implied_vol", "implied_vol_pct"]].head(30))

try:
    from IPython.display import display
    display(iv_uniform_df[["expiry", "T", "K", "price", "rate", "implied_vol", "implied_vol_pct"]])
except Exception:
    pass

In [ ]:
import json
import os
from math import erf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======================================================
# CONFIG
# ======================================================
DATASET_PATH = "data/gld_iv_dataset_200_600.pkl"
META_JSON_PATH = "data/gld_chain_wide_meta.json"
DIVIDEND_YIELD = 0.0

SIGMA_MIN = 1e-4
SIGMA_MAX = 3.0
GRID_SIZE = 2000   # più alto = più preciso
REFINE_STEPS = 4   # raffinamenti locali

# ======================================================
# INPUT
# ======================================================
if "iv_uniform_df" in globals():
    work_df = iv_uniform_df.copy()
else:
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"Dataset non trovato: {DATASET_PATH}")
    work_df = pd.read_pickle(DATASET_PATH)

required_cols = ["expiry", "T", "K", "price", "rate"]
missing = [c for c in required_cols if c not in work_df.columns]
if missing:
    raise RuntimeError(f"Mancano colonne necessarie nel dataset: {missing}")

# ======================================================
# SPOT DAL JSON
# ======================================================
def find_key_recursive(obj, candidate_keys):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in candidate_keys:
                return v
        for v in obj.values():
            out = find_key_recursive(v, candidate_keys)
            if out is not None:
                return out
    elif isinstance(obj, list):
        for item in obj:
            out = find_key_recursive(item, candidate_keys)
            if out is not None:
                return out
    return None

def load_spot_from_json(json_path):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"JSON metadata non trovato: {json_path}")

    with open(json_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    raw_spot = find_key_recursive(
        meta,
        ["S0", "spot", "spot_price", "underlying_price", "underlying_last"]
    )

    if raw_spot is None:
        raise RuntimeError("Non trovo lo spot nel JSON metadata.")

    S0 = float(raw_spot)
    if not np.isfinite(S0) or S0 <= 0:
        raise RuntimeError(f"Spot non valido: {S0}")

    return S0

S0 = load_spot_from_json(META_JSON_PATH)

# ======================================================
# BLACK-SCHOLES
# ======================================================
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

def bs_call_price(S, K, T, r, sigma, q=0.0):
    if T <= 0:
        return max(S - K, 0.0)

    if sigma <= 0:
        return max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)

    sqrtT = np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    return (
        S * np.exp(-q * T) * norm_cdf(d1)
        - K * np.exp(-r * T) * norm_cdf(d2)
    )

def bs_call_price_vec(S, K, T, r, sigma, q=0.0):
    K = np.asarray(K, dtype=float)
    T = np.asarray(T, dtype=float)
    r = np.asarray(r, dtype=float)

    sqrtT = np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    return (
        S * np.exp(-q * T) * np.vectorize(norm_cdf)(d1)
        - K * np.exp(-r * T) * np.vectorize(norm_cdf)(d2)
    )

# ======================================================
# OBIETTIVO: MEDIA ERRORE RELATIVO ASSOLUTO
# ======================================================
def mare_for_sigma(group, sigma, S0, q=0.0):
    K = group["K"].values.astype(float)
    T = group["T"].values.astype(float)
    r = group["rate"].values.astype(float)
    market = group["price"].values.astype(float)

    model = bs_call_price_vec(S0, K, T, r, sigma, q=q)
    rel_abs_err = np.abs(model - market) / market
    return float(np.mean(rel_abs_err))

def fit_best_sigma_for_expiry(group, S0, q=0.0,
                              sigma_min=SIGMA_MIN,
                              sigma_max=SIGMA_MAX,
                              grid_size=GRID_SIZE,
                              refine_steps=REFINE_STEPS):
    a = sigma_min
    b = sigma_max

    best_sigma = None
    best_mare = np.inf

    for _ in range(refine_steps):
        grid = np.linspace(a, b, grid_size)
        values = np.array([mare_for_sigma(group, s, S0, q=q) for s in grid])

        idx = int(np.argmin(values))
        best_sigma = float(grid[idx])
        best_mare = float(values[idx])

        if idx == 0:
            left = grid[0]
            right = grid[1]
        elif idx == len(grid) - 1:
            left = grid[-2]
            right = grid[-1]
        else:
            left = grid[idx - 1]
            right = grid[idx + 1]

        a, b = left, right

    return best_sigma, best_mare

# ======================================================
# STIMA SIGMA OTTIMO PER OGNI SCADENZA
# ======================================================
rows = []

for expiry, g in work_df.groupby("expiry", sort=True):
    g = g.sort_values("K").reset_index(drop=True)

    sigma_star, mare_star = fit_best_sigma_for_expiry(
        group=g,
        S0=S0,
        q=DIVIDEND_YIELD
    )

    model_prices = bs_call_price_vec(
        S=S0,
        K=g["K"].values,
        T=g["T"].values,
        r=g["rate"].values,
        sigma=sigma_star,
        q=DIVIDEND_YIELD
    )

    rel_abs_err = np.abs(model_prices - g["price"].values) / g["price"].values

    row = {
        "expiry": expiry,
        "T": float(g["T"].iloc[0]),
        "n_points": int(len(g)),
        "sigma_star": sigma_star,
        "sigma_star_pct": 100.0 * sigma_star,
        "MARE": mare_star,
        "MARE_pct": 100.0 * mare_star,
        "mean_market_price": float(g["price"].mean()),
        "mean_abs_rel_error_pct_check": 100.0 * float(rel_abs_err.mean())
    }

    if "implied_vol" in g.columns:
        row["mean_iv_pct"] = 100.0 * float(g["implied_vol"].mean())
        row["median_iv_pct"] = 100.0 * float(g["implied_vol"].median())

    rows.append(row)

sigma_table = pd.DataFrame(rows).sort_values("T").reset_index(drop=True)

# ======================================================
# OUTPUT TABELLA
# ======================================================
print(f"Spot usato: {S0:.6f}")
print()
print(sigma_table)

try:
    from IPython.display import display
    display(sigma_table)
except Exception:
    pass

In [ ]:
sigma_table_view = sigma_table[
    ["expiry", "T", "n_points", "sigma_star_pct", "MARE_pct", "mean_iv_pct", "median_iv_pct"]
].copy()

print(sigma_table_view)

try:
    display(sigma_table_view)
except Exception:
    pass